# Held-out representation diagnostics

## tl;dr

This notebook ranks the CRATER classes from weakest to strongest on untouched Humvee test data and displays the held-out images behind the mistakes. It covers both model layers: six-class part detection and grouped part-damage classification.

The ranking is ordered by F1 from low to high. Training support, held-out support, error counts, and coverage flags are shown beside each score so a weak class with little data can be distinguished from a weak class that has plenty of representation.

## Context & Methods

The detector is evaluated against the fixed `humvee_source6` test annotations. A prediction is matched to one ground-truth box when IoU is at least `IOU_THRESHOLD`; matches are selected greedily by descending prediction score within each image. Correct class matches are true positives. Missed boxes are false negatives, extra boxes are false positives, and overlapping boxes with the wrong class count as both a false negative for the true class and a false positive for the predicted class.

The damage models are evaluated on real test crops from the frozen damage run. Each crop is linked back to its original image and annotation box. Damage metrics are computed separately for each component group because the repository trains one classifier per group.

### Key Assumptions

- F1 is the primary weakness measure; lower F1 means weaker held-out performance.
- Classes with zero held-out support are reported as coverage gaps, not ranked as model failures.
- Small held-out support makes a score unstable, so the notebook flags low-support rows rather than over-interpreting them.
- Checkpoints and splits are loaded from existing outputs; this notebook does not train, tune, or change model selection.

## Data

### 1. Load inputs and parameters

In [ ]:
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'configs/taxonomy.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from within the CRATER repository.')
sys.path[:0] = [str(ROOT), str(ROOT / 'third_party/YOLOX')]

from yolox.data import ValTransform
from yolox.exp import get_exp
from yolox.utils import postprocess
from experiments.damage.training import (
    DamageCrops,
    TrainingConfig,
    build_model,
    make_loader,
    predict,
)
from tools.data.import_cvat_damage_annotations import PART_GROUPS, sha256_file

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DETECTION_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
LOW_SUPPORT = 10
LOW_TRAIN_SUPPORT = 20
MAX_IMAGES_PER_DETECTOR_CLASS = 2
MAX_IMAGES_PER_DAMAGE_CLASS = 2

TEST_ANN_PATH = ROOT / 'datasets/military/components/annotations/humvee_source6_instances_test.json'
DETECTOR_PATH = ROOT / 'outputs/detection/humvee_source6/carparts_initialized_seed42/best_ckpt.pth'
DAMAGE_RUN = ROOT / 'outputs/damage/humvee_cvat/humvee_damage_real_then_synthetic_v1'
CROP_DIR = DAMAGE_RUN / 'data'

try:
    font = ImageFont.truetype('arial.ttf', 15)
except OSError:
    font = ImageFont.load_default()

pd.set_option('display.max_columns', 50)
print(f'CRATER root: {ROOT}')
print(f'PyTorch {torch.__version__}; device={DEVICE}')
print(f'Detector threshold={DETECTION_THRESHOLD}; IoU threshold={IOU_THRESHOLD}')

In [ ]:
taxonomy = json.loads((ROOT / 'configs/taxonomy.json').read_text())
DAMAGE_LEVELS = taxonomy['damage']['levels']

test_coco = json.loads(TEST_ANN_PATH.read_text())
categories = sorted(test_coco['categories'], key=lambda row: row['id'])
PARTS = [row['name'] for row in categories]
PART_ID_TO_NAME = {row['id']: row['name'] for row in categories}
IMAGE_BY_ID = {}
GROUND_TRUTH_BY_IMAGE = defaultdict(list)

for image in test_coco['images']:
    image_path = ROOT / 'datasets/military/components' / image['file_name']
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    IMAGE_BY_ID[image['id']] = {**image, 'path': image_path}

for annotation in test_coco['annotations']:
    x, y, width, height = annotation['bbox']
    GROUND_TRUTH_BY_IMAGE[annotation['image_id']].append({
        'gt_id': annotation['id'],
        'class_id': annotation['category_id'],
        'class_name': PART_ID_TO_NAME[annotation['category_id']],
        'box': [float(x), float(y), float(x + width), float(y + height)],
    })

def annotation_counts(path):
    data = json.loads(path.read_text())
    names = {row['id']: row['name'] for row in data['categories']}
    counts = pd.Series([names[row['category_id']] for row in data['annotations']],
                       dtype='object').value_counts()
    return counts.reindex(PARTS, fill_value=0)

split_paths = {
    split: TEST_ANN_PATH.with_name(f'humvee_source6_instances_{split}.json')
    for split in ('train', 'val', 'test')
}
detector_coverage = pd.DataFrame({split: annotation_counts(path)
                                  for split, path in split_paths.items()})
detector_coverage.index.name = 'part_class'

crop_table = pd.read_csv(CROP_DIR / 'crops_manifest.csv')
required_crop_columns = {
    'annotation_id', 'source_image', 'component_class', 'component_group',
    'damage_level', 'domain', 'split', 'image_path', 'crop_file',
    'box_x1', 'box_y1', 'box_x2', 'box_y2'
}
missing_crop_columns = required_crop_columns - set(crop_table.columns)
if missing_crop_columns:
    raise ValueError(f'Crop manifest is missing columns: {sorted(missing_crop_columns)}')
damage_test_table = crop_table[(crop_table['domain'] == 'real')
                              & (crop_table['split'] == 'test')].copy()
if damage_test_table.empty:
    raise ValueError('No real held-out damage crops were found.')

test_box_count = len(test_coco['annotations'])
damage_source_count = damage_test_table['source_image'].nunique()
print(f'Detector test images: {len(IMAGE_BY_ID)}; ground-truth boxes: {test_box_count}')
print(f'Damage test crops: {len(damage_test_table)}; source images: {damage_source_count}')
display(detector_coverage)
display(damage_test_table.groupby(['component_group', 'damage_level']).size().unstack(fill_value=0))

## Results

### 2. Part detector: held-out predictions and matching

In [ ]:
exp = get_exp(str(ROOT / 'experiments/detection/yolox_s_crater6.py'), None)
if exp.num_classes != len(PARTS):
    raise ValueError(f'Detector config has {exp.num_classes} classes but annotations have {len(PARTS)}.')
detector = exp.get_model()
detector_checkpoint = torch.load(DETECTOR_PATH, map_location='cpu', weights_only=False)
detector.load_state_dict(detector_checkpoint['model'])
detector.to(DEVICE).eval()
detector_transform = ValTransform(legacy=False)

@torch.inference_mode()
def predict_parts(image_path):
    with Image.open(image_path) as source:
        original = source.convert('RGB')
    raw_bgr = cv2.cvtColor(np.asarray(original), cv2.COLOR_RGB2BGR)
    ratio = min(exp.test_size[0] / original.height, exp.test_size[1] / original.width)
    network_input, _ = detector_transform(raw_bgr, None, exp.test_size)
    output = detector(torch.from_numpy(network_input).unsqueeze(0).float().to(DEVICE))
    detections = postprocess(output, exp.num_classes, DETECTION_THRESHOLD, exp.nmsthre)[0]
    predictions = []
    if detections is None:
        return predictions
    for prediction_index, detection in enumerate(detections.cpu()):
        x1, y1, x2, y2 = (detection[:4] / ratio).tolist()
        box = [
            max(0.0, min(float(original.width), float(x1))),
            max(0.0, min(float(original.height), float(y1))),
            max(0.0, min(float(original.width), float(x2))),
            max(0.0, min(float(original.height), float(y2))),
        ]
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        class_index = int(detection[6])
        predictions.append({
            'pred_id': prediction_index,
            'class_id': class_index + 1,
            'class_name': PARTS[class_index],
            'box': box,
            'score': float(detection[4] * detection[5]),
            'objectness': float(detection[4]),
            'class_score': float(detection[5]),
        })
    return predictions

def box_iou(box_a, box_b):
    left = max(box_a[0], box_b[0])
    top = max(box_a[1], box_b[1])
    right = min(box_a[2], box_b[2])
    bottom = min(box_a[3], box_b[3])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0

def match_detections(ground_truth, predictions):
    remaining_ground_truth = set(range(len(ground_truth)))
    pairs, unmatched_predictions = [], []
    for prediction_index in sorted(range(len(predictions)),
                                   key=lambda index: predictions[index]['score'], reverse=True):
        if not remaining_ground_truth:
            unmatched_predictions.append(prediction_index)
            continue
        candidates = [(box_iou(predictions[prediction_index]['box'], ground_truth[index]['box']), index)
                      for index in remaining_ground_truth]
        best_iou, best_ground_truth_index = max(candidates)
        if best_iou >= IOU_THRESHOLD:
            remaining_ground_truth.remove(best_ground_truth_index)
            pairs.append((best_ground_truth_index, prediction_index, best_iou))
        else:
            unmatched_predictions.append(prediction_index)
    return pairs, sorted(remaining_ground_truth), unmatched_predictions

predictions_by_image = {}
detector_error_rows = []
detector_pair_rows = []
detector_counts = {name: {'tp': 0, 'fp': 0, 'fn': 0} for name in PARTS}

for image_id, image_record in IMAGE_BY_ID.items():
    ground_truth = GROUND_TRUTH_BY_IMAGE[image_id]
    predictions = predict_parts(image_record['path'])
    predictions_by_image[image_id] = predictions
    pairs, missing_ground_truth, extra_predictions = match_detections(ground_truth, predictions)
    for ground_truth_index, prediction_index, iou in pairs:
        gt = ground_truth[ground_truth_index]
        pred = predictions[prediction_index]
        detector_pair_rows.append({
            'image_id': image_id, 'file_name': image_record['file_name'],
            'true_class': gt['class_name'], 'predicted_class': pred['class_name'],
            'iou': iou, 'score': pred['score'],
        })
        if gt['class_name'] == pred['class_name']:
            detector_counts[gt['class_name']]['tp'] += 1
        else:
            detector_counts[gt['class_name']]['fn'] += 1
            detector_counts[pred['class_name']]['fp'] += 1
            for error_class, side in ((gt['class_name'], 'ground_truth'),
                                      (pred['class_name'], 'prediction')):
                detector_error_rows.append({
                    'image_id': image_id, 'file_name': image_record['file_name'],
                    'image_path': str(image_record['path']), 'error_class': error_class,
                    'error_type': 'wrong_class', 'side': side,
                    'true_class': gt['class_name'], 'predicted_class': pred['class_name'],
                    'gt_id': gt['gt_id'], 'pred_id': pred['pred_id'],
                    'gt_box': gt['box'], 'pred_box': pred['box'],
                    'iou': iou, 'score': pred['score'],
                })
    for ground_truth_index in missing_ground_truth:
        gt = ground_truth[ground_truth_index]
        detector_counts[gt['class_name']]['fn'] += 1
        detector_error_rows.append({
            'image_id': image_id, 'file_name': image_record['file_name'],
            'image_path': str(image_record['path']), 'error_class': gt['class_name'],
            'error_type': 'missed_ground_truth', 'side': 'ground_truth',
            'true_class': gt['class_name'], 'predicted_class': None,
            'gt_id': gt['gt_id'], 'pred_id': None, 'gt_box': gt['box'],
            'pred_box': None, 'iou': 0.0, 'score': None,
        })
    for prediction_index in extra_predictions:
        pred = predictions[prediction_index]
        detector_counts[pred['class_name']]['fp'] += 1
        detector_error_rows.append({
            'image_id': image_id, 'file_name': image_record['file_name'],
            'image_path': str(image_record['path']), 'error_class': pred['class_name'],
            'error_type': 'extra_prediction', 'side': 'prediction',
            'true_class': None, 'predicted_class': pred['class_name'],
            'gt_id': None, 'pred_id': pred['pred_id'], 'gt_box': None,
            'pred_box': pred['box'], 'iou': 0.0, 'score': pred['score'],
        })

detector_error_table = pd.DataFrame(detector_error_rows)
detector_pair_table = pd.DataFrame(detector_pair_rows)

def metric_row(name, counts):
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'class_name': name, 'test_support': tp + fn, 'tp': tp, 'fp': fp, 'fn': fn,
            'errors': fp + fn, 'precision': precision, 'recall': recall, 'f1': f1}

detector_ranking = pd.DataFrame([metric_row(name, detector_counts[name]) for name in PARTS])
detector_ranking = detector_ranking.merge(detector_coverage[['train', 'val', 'test']],
                                           left_on='class_name', right_index=True)
detector_ranking = detector_ranking.rename(columns={'train': 'train_examples', 'test': 'test_annotations'})
detector_ranking['coverage_flag'] = np.where(
    detector_ranking['test_support'] < LOW_SUPPORT, 'low held-out support',
    np.where(detector_ranking['train_examples'] < LOW_TRAIN_SUPPORT, 'low train support', 'adequate support'))
detector_ranking = detector_ranking.sort_values(['f1', 'recall', 'test_support'],
                                                ascending=[True, True, False]).reset_index(drop=True)
detector_ranking.insert(0, 'performance_rank', np.arange(1, len(detector_ranking) + 1))

print(f'Predicted {sum(len(rows) for rows in predictions_by_image.values())} boxes across {len(predictions_by_image)} held-out images.')
wrong_class_pair_count = (detector_error_table['error_type'] == 'wrong_class').sum() // 2 if not detector_error_table.empty else 0
print(f'Error events: {len(detector_error_table)}; wrong-class pairs: {wrong_class_pair_count}')

### 3. Ranked part-detector performance

This is the requested weakest-to-strongest class list for the detector. `errors` counts false positives plus false negatives, so it exposes the underlying error volume alongside F1.

In [ ]:
detector_rank_view = detector_ranking[[
    'performance_rank', 'class_name', 'f1', 'precision', 'recall',
    'train_examples', 'test_support', 'errors', 'tp', 'fp', 'fn', 'coverage_flag'
]].copy()
def f1_bar(value, width=20):
    if pd.isna(value):
        return 'n/a'
    filled = max(0, min(width, round(float(value) * width)))
    return '#' * filled + '-' * (width - filled)

detector_rank_view.insert(3, 'f1_bar', detector_rank_view['f1'].map(f1_bar))
display(detector_rank_view.round({'f1': 3, 'precision': 3, 'recall': 3}))

if detector_error_table.empty:
    print('No detector error events at the configured threshold.')
else:
    display(detector_error_table.groupby(['error_class', 'error_type']).size()
            .rename('error_events').reset_index().sort_values(['error_class', 'error_events'],
                                                                  ascending=[True, False]))

### 4. Visualize detector mistakes on held-out images

In [ ]:
def scaled_box(box, sx, sy):
    return [box[0] * sx, box[1] * sy, box[2] * sx, box[3] * sy]

def draw_box_label(draw, box, label, color, width=3):
    draw.rectangle(box, outline=color, width=width)
    text_box = draw.textbbox((0, 0), label, font=font)
    text_width = text_box[2] - text_box[0]
    text_height = text_box[3] - text_box[1]
    x = max(0, min(box[0], draw.im.size[0] - text_width - 6))
    y = max(0, box[1] - text_height - 5)
    draw.rectangle((x, y, x + text_width + 5, y + text_height + 3), fill='white')
    draw.text((x + 2, y + 1), label, fill=color, font=font)

def render_detector_error(image_id, focus_class):
    record = IMAGE_BY_ID[image_id]
    with Image.open(record['path']) as source:
        original = source.convert('RGB')
    preview = original.copy()
    preview.thumbnail((1100, 760))
    sx, sy = preview.width / original.width, preview.height / original.height
    draw = ImageDraw.Draw(preview)
    focused_errors = detector_error_table[(detector_error_table['image_id'] == image_id)
                                         & (detector_error_table['error_class'] == focus_class)]
    error_gt_ids = set(focused_errors['gt_id'].dropna().astype(int))
    error_pred_ids = set(focused_errors['pred_id'].dropna().astype(int))
    for gt in GROUND_TRUTH_BY_IMAGE[image_id]:
        is_error = gt['gt_id'] in error_gt_ids
        color = '#e74c3c' if is_error else '#229954'
        label = f"GT: {gt['class_name']}"
        draw_box_label(draw, scaled_box(gt['box'], sx, sy), label, color, 4 if is_error else 2)
    for pred in predictions_by_image[image_id]:
        is_error = pred['pred_id'] in error_pred_ids
        color = '#e74c3c' if is_error else '#2874a6'
        label = f"P: {pred['class_name']} {pred['score']:.2f}"
        draw_box_label(draw, scaled_box(pred['box'], sx, sy), label, color, 4 if is_error else 2)
    return preview

def show_detector_error_gallery(max_images=MAX_IMAGES_PER_DETECTOR_CLASS):
    if detector_error_table.empty:
        print('No detector mistakes to visualize.')
        return
    for focus_class in detector_ranking['class_name']:
        class_errors = detector_error_table[detector_error_table['error_class'] == focus_class]
        image_ids = list(dict.fromkeys(class_errors['image_id'].tolist()))[:max_images]
        print(f'{focus_class}: {len(class_errors)} error events; showing {len(image_ids)} image(s).')
        for image_id in image_ids:
            display(render_detector_error(image_id, focus_class))

show_detector_error_gallery()

### 5. Damage classification: held-out predictions and ranking

In [ ]:
selection = json.loads((DAMAGE_RUN / 'selected_checkpoints.json').read_text())
damage_predictions = []
damage_metrics = {}
damage_models = {}

for group in sorted(set(PART_GROUPS.values())):
    checkpoint_record = selection[group]
    checkpoint_path = DAMAGE_RUN / checkpoint_record['path'].replace('\\', '/')
    if sha256_file(checkpoint_path) != checkpoint_record['sha256']:
        raise ValueError(f'Changed damage checkpoint: {checkpoint_path}')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    levels = checkpoint['levels']
    if levels != DAMAGE_LEVELS:
        raise ValueError(f'Unexpected damage label order for {group}: {levels}')
    model = build_model(len(levels), initialization='scratch')
    model.load_state_dict(checkpoint['model'])
    model.to(DEVICE).eval()
    damage_models[group] = model
    group_rows = damage_test_table[damage_test_table['component_group'] == group].to_dict('records')
    if not group_rows:
        continue
    loader = make_loader(group_rows, CROP_DIR, levels,
                        TrainingConfig(**checkpoint['config']))
    metrics, probabilities = predict(model, loader, DEVICE, levels)
    damage_metrics[group] = metrics
    for row, probability in zip(group_rows, probabilities):
        probability = np.asarray(probability)
        prediction_index = int(probability.argmax())
        damage_predictions.append({
            **row, 'prediction': levels[prediction_index],
            'damage_score': float(probability[prediction_index]),
            **{f'p_{level}': float(value) for level, value in zip(levels, probability)},
        })

damage_prediction_table = pd.DataFrame(damage_predictions)
damage_prediction_table['is_correct'] = (damage_prediction_table['damage_level']
                                          == damage_prediction_table['prediction'])
damage_error_table = damage_prediction_table[~damage_prediction_table['is_correct']].copy()

damage_train_counts = (crop_table[(crop_table['domain'] == 'real') & (crop_table['split'] == 'train')]
                       .groupby(['component_group', 'damage_level']).size()
                       .rename('train_examples'))

damage_rank_rows = []
for group in sorted(set(PART_GROUPS.values())):
    group_predictions = damage_prediction_table[damage_prediction_table['component_group'] == group]
    for level in DAMAGE_LEVELS:
        true = group_predictions['damage_level'] == level
        predicted = group_predictions['prediction'] == level
        tp = int((true & predicted).sum())
        fp = int((~true & predicted).sum())
        fn = int((true & ~predicted).sum())
        support = int(true.sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / support if support else np.nan
        f1 = (2 * precision * recall / (precision + recall)
              if precision + recall and support else 0.0 if support else np.nan)
        damage_rank_rows.append({
            'group': group, 'damage_class': level, 'test_support': support,
            'errors': fp + fn, 'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall, 'f1': f1,
            'train_examples': int(damage_train_counts.get((group, level), 0)),
        })

damage_ranking_all = pd.DataFrame(damage_rank_rows)
damage_ranking_all['coverage_flag'] = np.select(
    [damage_ranking_all['test_support'].eq(0),
     damage_ranking_all['test_support'].lt(LOW_SUPPORT),
     damage_ranking_all['train_examples'].lt(LOW_TRAIN_SUPPORT)],
    ['no held-out support', 'low held-out support', 'low train support'],
    default='adequate support')
damage_ranking = (damage_ranking_all[damage_ranking_all['test_support'] > 0]
                  .sort_values(['f1', 'recall', 'test_support'],
                               ascending=[True, True, False]).reset_index(drop=True))
damage_ranking.insert(0, 'performance_rank', np.arange(1, len(damage_ranking) + 1))
damage_coverage_gaps = damage_ranking_all[damage_ranking_all['test_support'] == 0].copy()

print(f'Evaluated {len(damage_prediction_table)} held-out damage crops; errors: {len(damage_error_table)}.')
display(pd.DataFrame([
    {'group': group, 'samples': metrics['samples'], 'accuracy': metrics['accuracy'],
     'macro_f1': metrics['macro_f1'], 'ece_10_bins': metrics['ece_10_bins']}
    for group, metrics in damage_metrics.items()
]).round(3))

### 6. Ranked damage-class performance

These rows are grouped as `component_group / damage_class` because the damage classifiers are group-specific. Rows with no held-out examples are listed separately as coverage gaps.

In [ ]:
damage_rank_view = damage_ranking[[
    'performance_rank', 'group', 'damage_class', 'f1', 'precision', 'recall',
    'train_examples', 'test_support', 'errors', 'tp', 'fp', 'fn', 'coverage_flag'
]].copy()
damage_rank_view.insert(4, 'f1_bar', damage_rank_view['f1'].map(f1_bar))
display(damage_rank_view.round({'f1': 3, 'precision': 3, 'recall': 3}))

if not damage_coverage_gaps.empty:
    print('Coverage gaps: these group/class combinations had no held-out test crops and cannot receive a performance rank.')
    display(damage_coverage_gaps[['group', 'damage_class', 'train_examples', 'test_support', 'coverage_flag']])


### 7. Visualize damage mistakes on held-out images

In [ ]:
def render_damage_error(row):
    source_path = ROOT / row['image_path']
    with Image.open(source_path) as source:
        original = source.convert('RGB')
    preview = original.copy()
    preview.thumbnail((900, 650))
    sx, sy = preview.width / original.width, preview.height / original.height
    draw = ImageDraw.Draw(preview)
    related = damage_test_table[damage_test_table['source_image'] == row['source_image']]
    focus_id = row['annotation_id']
    for _, related_row in related.iterrows():
        box = [float(related_row['box_x1']), float(related_row['box_y1']),
               float(related_row['box_x2']), float(related_row['box_y2'])]
        focused = related_row['annotation_id'] == focus_id
        color = '#e74c3c' if focused else '#7f8c8d'
        label = (f"GT {row['damage_level']} | P {row['prediction']}"
                 if focused else str(related_row['component_class']))
        draw_box_label(draw, scaled_box(box, sx, sy), label, color, 4 if focused else 2)
    crop_box = (int(float(row['box_x1'])), int(float(row['box_y1'])),
                int(float(row['box_x2'])), int(float(row['box_y2'])))
    crop = original.crop(crop_box)
    crop.thumbnail((320, 320))
    canvas_height = max(preview.height, 380)
    canvas = Image.new('RGB', (preview.width + 360, canvas_height), 'white')
    canvas.paste(preview, (0, 0))
    canvas.paste(crop, (preview.width + 20, 45))
    canvas_draw = ImageDraw.Draw(canvas)
    canvas_draw.text((preview.width + 20, 12), 'Held-out crop', fill='#17202a', font=font)
    canvas_draw.text((preview.width + 20, min(canvas_height - 28, 350)),
                     f"{row['component_group']} / {row['component_class']}",
                     fill='#17202a', font=font)
    return canvas

def show_damage_error_gallery(max_images=MAX_IMAGES_PER_DAMAGE_CLASS):
    if damage_error_table.empty:
        print('No damage-classification mistakes to visualize.')
        return
    for _, rank_row in damage_ranking.iterrows():
        subset = damage_error_table[(damage_error_table['component_group'] == rank_row['group'])
                                     & ((damage_error_table['damage_level'] == rank_row['damage_class'])
                                        | (damage_error_table['prediction'] == rank_row['damage_class']))]
        subset = subset.drop_duplicates('source_image').head(max_images)
        print(f"{rank_row['group']} / {rank_row['damage_class']}: {int(rank_row['errors'])} error(s); showing {len(subset)} image(s).")
        for _, row in subset.iterrows():
            display(render_damage_error(row))

show_damage_error_gallery()

## Takeaways

Use the lowest-F1 rows and their galleries to decide what to label next. A practical collection priority is: first cover group/class combinations with no held-out support, then add examples for low-F1 classes with low training support, then inspect low-F1 classes that already have substantial support for annotation quality, ambiguity, or model limitations.

In [ ]:
def collection_signal(row):
    if row['test_support'] == 0:
        return 'coverage gap: collect and hold out examples'
    if row['f1'] < 0.50 and row['train_examples'] < LOW_TRAIN_SUPPORT:
        return 'highest priority: add representation and review labels'
    if row['f1'] < 0.75 and row['train_examples'] < LOW_TRAIN_SUPPORT:
        return 'add representation; weak and low support'
    if row['f1'] < 0.75:
        return 'review errors; weakness persists despite support'
    if row['test_support'] < LOW_SUPPORT:
        return 'monitor: score is support-limited'
    return 'monitor'

detector_priority = detector_ranking.copy()
detector_priority['collection_signal'] = detector_priority.apply(collection_signal, axis=1)
print('Detector collection priorities')
display(detector_priority[['performance_rank', 'class_name', 'f1', 'train_examples',
                           'test_support', 'errors', 'collection_signal']]
        .round({'f1': 3}))

damage_priority = damage_ranking.copy()
damage_priority['collection_signal'] = damage_priority.apply(collection_signal, axis=1)
print('Damage collection priorities among evaluated combinations')
display(damage_priority[['performance_rank', 'group', 'damage_class', 'f1',
                        'train_examples', 'test_support', 'errors', 'collection_signal']]
        .round({'f1': 3}))

print('Interpretation caveat: rankings describe this fixed held-out split and threshold. Validate proposed new labels with a fresh split after collection.')